# V-JEPA 2 on LaSOT (Kaggle)

1. **Re-identification check** (`scripts/vjepa_reid_check.py`): can V-JEPA 2 features re-find the target after an occlusion? ~250 episodes of the dev subset, motion clip vs repeated frame. Small output.
2. **Feature cache** (`scripts/cache_vjepa.py`) for the 40 dev videos, one map every 8 frames (~7 GB).

Settings: **Accelerator = GPU T4 x2**, **Internet = On**. Then **Save Version -> Save & Run All (Commit)**.
Set `RUN_CACHE = False` below to run only the check (much faster).

In [ ]:
# fail fast with a clear message if the session has no internet
import socket
try:
    socket.create_connection(('github.com', 443), timeout=10).close()
except OSError:
    raise SystemExit('NO INTERNET: Settings (right panel) -> Internet = On '
                     '(needs a phone-verified Kaggle account), then run again.')


In [ ]:
RUN_CACHE = True
import subprocess  # check=True stops the run here if cloning fails
subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/MariiaHamanuk/Object-Tracking.git',
                '/kaggle/working/Object-Tracking'], check=True)
!pip install -q -U "transformers>=4.53" gdown
!nvidia-smi --query-gpu=name,memory.total --format=csv
# official OSTrack results on LaSOT (to know where the tracker went after each occlusion)
!mkdir -p /kaggle/tmp && gdown 1Cx1WMwArkmZcKJNx5rYEBQmm1G65q0Ws -O /kaggle/tmp/ostrack_lasot.zip && unzip -qo /kaggle/tmp/ostrack_lasot.zip -d /kaggle/tmp/ostrack

In [ ]:
%cd /kaggle/working/Object-Tracking
!python scripts/vjepa_reid_check.py --tracker /kaggle/tmp/ostrack/lasot --work /kaggle/tmp/reid --out /kaggle/working/results/vjepa_reid_check 2>&1 | grep -v Warning | tail -n 40

In [ ]:
import subprocess, torch, os
if RUN_CACHE:
    n = max(torch.cuda.device_count(), 1)
    procs = []
    for i in range(n):
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(i))
        log = open(f'/kaggle/working/vjepa_cache_shard{i}.log', 'w')
        procs.append(subprocess.Popen(
            ['python', 'scripts/cache_vjepa.py', '--out', '/kaggle/working/cache/vjepa', '--work', '/kaggle/tmp',
             '--shard', str(i), '--num-shards', str(n)], env=env, stdout=log, stderr=subprocess.STDOUT))
    print([p.wait() for p in procs])

In [ ]:
!tail -n 3 /kaggle/working/vjepa_cache_shard*.log 2>/dev/null
!cd /kaggle/working && zip -qr vjepa_reid_check.zip results && rm -rf Object-Tracking && du -sh /kaggle/working/*